# 01 — GCaMP batch processing from Suite2p output

This notebook processes Suite2p calcium-imaging output together with VR behaviour files. It keeps the core batch-processing workflow from the original analysis notebook, while removing later exploratory sections.

The notebook is intended for publication/repository use:

- paths and settings are collected at the top;
- reusable analysis functions are imported from `helper_functions.py`;
- each major processing step is separated into a clearly labelled section;
- outputs are saved per mouse and per session.

## 1. Imports and repository paths

This section imports standard scientific Python packages and adds the repository `functions/` directory to the Python path. This allows the notebook to use `helper_functions.py` after the repository has been cloned, without needing absolute paths on the user's machine.

In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from joblib import Parallel, delayed
from scipy import stats
from scipy.ndimage import gaussian_filter
from scipy.io import loadmat

# -------------------------------------------------------------------------
# Locate repository folders
# -------------------------------------------------------------------------
# Expected repository layout:
#
#   GCaMP/
#   ├── notebooks/
#   │   └── 01_GCaMP_batch_suite2p_place_cells.ipynb
#   └── functions/
#       └── helper_functions.py
#
# If this notebook is somewhere else, edit REPO_ROOT manually.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() in {"notebooks", "main"} else NOTEBOOK_DIR

FUNCTIONS_DIR = REPO_ROOT / "functions"
if FUNCTIONS_DIR.exists():
    sys.path.insert(0, str(FUNCTIONS_DIR))

# Import reusable analysis functions from helper_functions.py
from helper_functions import (
    baseline_correction,
    get_significant_transients,
    calculate_mad_snr,
    bin_data_trace,
    get_spatial_information,
    shuffle_transient_trace,
    calculate_placecell_pcnt,
)

## 2. User settings

Edit this section for each dataset. `BASE_DIR` should point to the folder containing one folder per mouse. Each mouse folder is expected to contain Suite2p output files (`F.npy`, `Fneu.npy`, `iscell.npy`) and the processed behaviour `.mat` files.

In [ ]:
# -------------------------------------------------------------------------
# Dataset location
# -------------------------------------------------------------------------
BASE_DIR = Path(r"C:\path\to\VR Analysis\Cohort 7\data")
DATE = "250408"
MICE = ["DG31", "DG32", "DG33", "DG35", "DG36"]

# -------------------------------------------------------------------------
# Behaviour file names
# -------------------------------------------------------------------------
# These are formatted for each mouse and date.
FULL_BEHAV_FILE_FMT = "{mouse}_{date}_tyv.mat"
PRE_BEHAV_FILE_FMT  = "{mouse}_{date}_pre_tyv.mat"
POST_BEHAV_FILE_FMT = "{mouse}_{date}_post_tyv.mat"

# -------------------------------------------------------------------------
# Output folders created inside each mouse folder
# -------------------------------------------------------------------------
OUT_PRE  = "output_data_pre_fix"
OUT_POST = "output_data_post_fix"

# Re-run analyses even if output files already exist.
OVERWRITE = False

## 3. Analysis parameters

These values control preprocessing, transient extraction, spatial binning, and the place-cell shuffle test. They are collected here so that the analysis can be reproduced and modified without searching through the notebook.

In [ ]:
# Suite2p neuropil subtraction coefficient
NEUROPIL_COEFF = 0.7

# Offset added before baseline correction to avoid negative fluorescence values
FLUORESCENCE_OFFSET = 15000

# Baseline correction settings used by helper_functions.baseline_correction
BASELINE_WINDOW_FRAMES = 1800
BASELINE_PERCENTILE = 0.08

# SNR threshold used to remove anomalously large traces after transient extraction
SNR_THRESHOLD = 120

# Spatial tuning settings
TRACK_START_CM = 0
TRACK_LENGTH_CM = 295
N_POSITION_BINS = 59
VELOCITY_THRESHOLD_CM_S = 5
GAUSSIAN_SIGMA_BINS = 2

# Place-cell significance test
N_SHUFFLES = 1000
PLACE_CELL_ALPHA = 0.05

# Behaviour/session settings
SESSIONS = ("pre", "post")

## 4. Plotting helper

This function saves a sorted ratemap heatmap for significant place cells. It is kept in the notebook because it is specific to this workflow and output style, whereas the general-purpose analysis operations remain in `helper_functions.py`.

In [ ]:
def save_ratemap_figure(
    ratemaps_norm,
    save_path,
    session,
    mouse,
    reward_pos=None,
    track_length_cm=TRACK_LENGTH_CM,
    number_bins=None,
    fontsize=12,
    filename="place_cell_ratemaps.png",
    cmap="viridis",
    show=False,
):
    """Save a sorted heatmap of normalised place-cell ratemaps."""

    save_path = Path(save_path)
    save_path.mkdir(parents=True, exist_ok=True)

    if ratemaps_norm is None or getattr(ratemaps_norm, "size", 0) == 0:
        print(f"[FIG] No ratemaps to plot for {mouse} {session}")
        return

    ratemaps_norm = np.asarray(ratemaps_norm)
    n_cells, n_bins = ratemaps_norm.shape
    if number_bins is None:
        number_bins = n_bins

    # Sort cells by peak spatial bin.
    peak_idx = np.argmax(ratemaps_norm, axis=1)
    sort_idx = np.argsort(peak_idx)
    sorted_rates = ratemaps_norm[sort_idx]

    # Convert reward positions from cm to bin coordinates.
    reward_bin = None
    if reward_pos is not None and len(np.atleast_1d(reward_pos)) > 0:
        reward_pos = np.atleast_1d(reward_pos).astype(float)
        reward_bin = reward_pos / (track_length_cm / number_bins)

    fig, ax = plt.subplots(1, 1, figsize=(3, 6))

    sns.heatmap(
        sorted_rates,
        cmap=cmap,
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={"pad": 0.15, "ticks": [0, 1]},
    )

    ax.set_title(f"{mouse} {session.upper()} place cells", fontsize=fontsize + 2, pad=20)
    ax.set_xlabel("Track position (bins)", fontsize=fontsize + 2)
    ax.set_ylabel("Position-encoding cells", fontsize=fontsize + 2)

    step = max(1, round(number_bins / 2))
    ax.set_xticks(np.arange(0, number_bins + 1, step))
    ax.set_xticklabels(np.arange(0, number_bins + 1, step), rotation=0, fontsize=fontsize)

    # Show first and last cell labels only.
    ax.set_ylim(n_cells, 0)
    ax.set_yticks([0.5, n_cells - 0.5])
    ax.set_yticklabels([1, n_cells], rotation=0, fontsize=fontsize)
    ax.tick_params(length=0)

    if reward_bin is not None:
        colour = "white" if session == "pre" else "magenta"
        ax.arrow(float(np.min(reward_bin)), 0, 0, n_cells, color=colour, linewidth=1)
        ax.arrow(float(np.max(reward_bin)), 0, 0, n_cells, color=colour, linewidth=1)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=fontsize - 1, length=0)
    cbar.set_label("normalised ΔF/F", labelpad=-2, fontsize=fontsize - 1)

    plt.tight_layout()
    outpath = save_path / filename
    plt.savefig(outpath, dpi=300)
    print(f"[FIG] Saved {outpath}")

    if show:
        plt.show()

    plt.close(fig)

## 5. File-loading and alignment helpers

These helper functions load Suite2p arrays and behaviour files, compute the imaging frame interval from the full session, and align behaviour variables onto the two-photon frame times. The pre/post sessions are trimmed from the beginning or end of the Suite2p trace using the duration of the matching behaviour file.

In [ ]:
def mouse_dir(mouse: str) -> Path:
    """Return the folder for one mouse."""
    return BASE_DIR / DATE / mouse


def output_dir(mouse: str, session: str) -> Path:
    """Return the output folder for a given mouse and session."""
    if session == "pre":
        return mouse_dir(mouse) / OUT_PRE
    if session == "post":
        return mouse_dir(mouse) / OUT_POST
    raise ValueError("session must be 'pre' or 'post'")


def behav_file(mouse: str, session: str) -> str:
    """Return the expected behaviour filename for full, pre, or post sessions."""
    if session == "full":
        return FULL_BEHAV_FILE_FMT.format(mouse=mouse, date=DATE)
    if session == "pre":
        return PRE_BEHAV_FILE_FMT.format(mouse=mouse, date=DATE)
    if session == "post":
        return POST_BEHAV_FILE_FMT.format(mouse=mouse, date=DATE)
    raise ValueError("session must be 'full', 'pre', or 'post'")


def load_suite2p(mouse: str):
    """Load Suite2p fluorescence, neuropil fluorescence, and iscell arrays."""
    d = mouse_dir(mouse)
    F = np.load(d / "F.npy")
    Fneu = np.load(d / "Fneu.npy")
    iscell = np.load(d / "iscell.npy")
    return np.asarray(F), np.asarray(Fneu), np.asarray(iscell)


def compute_frame_interval(mouse: str) -> float:
    """Compute seconds per imaging frame from full-session behaviour and Suite2p frame count."""
    F, Fneu, iscell = load_suite2p(mouse)

    iscell_logic = iscell[:, 0].astype(bool)
    F_cell = F[iscell_logic]
    Fneu_cell = Fneu[iscell_logic]
    F1 = F_cell - NEUROPIL_COEFF * Fneu_cell

    behaviour_data = loadmat(mouse_dir(mouse) / behav_file(mouse, "full"), simplify_cells=True)
    timg = np.asarray(behaviour_data["timg"])

    n_frames = int(F1.shape[1])
    session_duration = float(timg[-1] - timg[0])
    frame_rate = n_frames / session_duration
    frame_interval = 1.0 / frame_rate

    print(f"{mouse}: frame rate = {frame_rate:.3f} Hz ({frame_interval:.5f} s/frame)")
    return frame_interval


def get_behaviour_arrays(behaviour_data: dict, session: str):
    """Extract behaviour arrays for pre or post sessions from the loaded .mat file."""
    if session == "pre":
        return (
            np.asarray(behaviour_data["t_pre"]),
            np.asarray(behaviour_data["y_pre"]),
            np.asarray(behaviour_data["v_pre"]),
            np.asarray(behaviour_data["reward_pre"]),
        )
    if session == "post":
        return (
            np.asarray(behaviour_data["t_post"]),
            np.asarray(behaviour_data["y_post"]),
            np.asarray(behaviour_data["v_post"]),
            np.asarray(behaviour_data["reward_post"]),
        )
    raise ValueError("session must be 'pre' or 'post'")


def load_and_align_session(mouse: str, session: str, frame_interval: float):
    """Load Suite2p and behaviour data, then align behaviour variables to imaging frames."""
    assert session in ("pre", "post")

    data_path = mouse_dir(mouse)
    behaviour_data = loadmat(data_path / behav_file(mouse, session), simplify_cells=True)

    F, Fneu, iscell = load_suite2p(mouse)

    # Generate stable cell identifiers based on Suite2p row number.
    n_all_cells = F.shape[0]
    width = len(str(n_all_cells - 1))
    cell_ids = np.array([f"C{str(i).zfill(width)}" for i in range(n_all_cells)])

    # Keep Suite2p cells and subtract neuropil.
    iscell_logic = iscell[:, 0].astype(bool)
    F_cell = F[iscell_logic]
    Fneu_cell = Fneu[iscell_logic]
    F1 = F_cell - (NEUROPIL_COEFF * Fneu_cell)
    cell_ids = cell_ids[iscell_logic]

    # Extract behaviour arrays.
    behave_time, behave_pos, behave_vel, behave_rew = get_behaviour_arrays(behaviour_data, session)
    behave_time_zero = behave_time - behave_time[0]

    # Trim the full Suite2p trace to match pre or post behaviour duration.
    n_frames_to_keep = int(np.floor(behave_time_zero[-1] / frame_interval))
    if session == "pre":
        F1 = F1[:, :n_frames_to_keep]
        print(f"{mouse} {session}: cut to first {n_frames_to_keep} imaging frames")
    else:
        F1 = F1[:, -n_frames_to_keep:]
        print(f"{mouse} {session}: cut to last {n_frames_to_keep} imaging frames")

    # Create an imaging-frame timebase covering the behaviour session.
    trace_length = F1.shape[1]
    recording_length = float(behave_time_zero[-1])
    trace_time = np.linspace(0, recording_length, trace_length)

    # Interpolate behaviour onto imaging-frame timestamps.
    coords = np.interp(trace_time, behave_time_zero, behave_pos)
    velocity = np.interp(trace_time, behave_time_zero, behave_vel)

    # Convert reward events to binary reward-onset trace in imaging-frame time.
    beh_rew = np.asarray(behave_rew).astype(float)
    beh_edges = np.where((beh_rew[1:] > 0.5) & (beh_rew[:-1] <= 0.5))[0] + 1
    reward_times = behave_time_zero[beh_edges]
    reward_frames = np.searchsorted(trace_time, reward_times, side="left")
    reward_frames = np.clip(reward_frames, 0, trace_time.size - 1)

    reward_trace = np.zeros(trace_time.shape[0], dtype=int)
    reward_trace[reward_frames] = 1

    # Normalise spatial coordinate to start at zero and keep only the track range.
    coords = coords - np.nanmin(coords)
    valid_mask = (coords >= TRACK_START_CM) & (coords <= TRACK_LENGTH_CM)

    coords = coords[valid_mask]
    trace_time = trace_time[valid_mask]
    velocity = velocity[valid_mask]
    reward_trace = reward_trace[valid_mask]
    F1 = F1[:, valid_mask]

    print(f"{mouse} {session}: kept {coords.shape[0]} frames after track clipping")

    return F1, coords, velocity, reward_trace, trace_time, cell_ids

## 6. Core place-cell pipeline

This section implements the main analysis for one mouse and one session. It performs baseline correction, extracts significant calcium transients, removes anomalously high-SNR traces, computes occupancy-normalised spatial activity maps, and estimates place-cell significance by comparing each cell's spatial information to shuffled transient traces.

In [ ]:
def compute_place_cell_metrics(Fc3_cleaned, coords, velocity, trace_time):
    """Compute spatial information, shuffle p-values, and ratemaps for all cells."""

    time_per_frame = float(trace_time.max() / trace_time.shape[0])
    velocity_mask = (velocity > VELOCITY_THRESHOLD_CM_S).astype(bool)
    coords_active = coords[velocity_mask]

    bin_edges = np.linspace(TRACK_START_CM, TRACK_LENGTH_CM, N_POSITION_BINS + 1)

    frames_per_bin, _ = np.histogram(coords_active, bins=bin_edges)
    time_per_bin = frames_per_bin * time_per_frame

    n_cells = Fc3_cleaned.shape[0]

    si_values = np.zeros(n_cells)
    avg_rate_values = np.zeros(n_cells)
    z_scores = np.zeros(n_cells)
    p_values = np.ones(n_cells)
    ratemaps = np.zeros((n_cells, N_POSITION_BINS))
    ratemaps_norm = np.zeros((n_cells, N_POSITION_BINS))

    smooth_time_binned = gaussian_filter(
        time_per_bin.astype(float),
        sigma=GAUSSIAN_SIGMA_BINS,
        truncate=2,
    ).reshape(-1, 1)
    smooth_time_binned_safe = np.maximum(smooth_time_binned, 1e-12)

    for i in tqdm(range(n_cells), desc="place-cell shuffle test"):
        trace = Fc3_cleaned[i, :]
        trace_active = trace[velocity_mask]

        # Skip cells with no transient activity during running.
        if np.sum(trace_active) == 0:
            continue

        # Bin calcium activity by position during running frames.
        binned_mean_dff = bin_data_trace(coords_active, trace_active, bin_edges)

        if np.max(binned_mean_dff) == 0:
            continue

        smooth_binned_mean_dff = gaussian_filter(
            binned_mean_dff.astype(float),
            sigma=GAUSSIAN_SIGMA_BINS,
            truncate=2,
        )

        # Occupancy-normalised spatial activity map.
        binned_ratemap = smooth_binned_mean_dff / smooth_time_binned_safe

        true_spatial_info, true_avg_rate = get_spatial_information(
            time_per_bin,
            binned_ratemap + 1e-10,
        )

        # Shuffle the transient trace and recompute spatial information.
        si_dist = []
        for _ in range(N_SHUFFLES):
            shuff_trace = shuffle_transient_trace(trace_active)

            binned_mean_dff_shuff = bin_data_trace(coords_active, shuff_trace, bin_edges)
            smooth_binned_mean_dff_shuff = gaussian_filter(
                binned_mean_dff_shuff.astype(float),
                sigma=GAUSSIAN_SIGMA_BINS,
                truncate=2,
            )

            binned_ratemap_shuff = smooth_binned_mean_dff_shuff / smooth_time_binned_safe

            shuff_si, _ = get_spatial_information(
                time_per_bin,
                binned_ratemap_shuff + 1e-10,
            )
            si_dist.append(shuff_si)

        pop_mean = float(np.mean(si_dist))
        pop_std = float(np.std(si_dist))
        if pop_std <= 0:
            pop_std = 1e-12

        z_score = float((true_spatial_info - pop_mean) / pop_std)
        p_value = float(stats.norm.sf(z_score))

        si_values[i] = true_spatial_info
        avg_rate_values[i] = true_avg_rate
        z_scores[i] = z_score
        p_values[i] = p_value
        ratemaps[i, :] = binned_ratemap.ravel()

        mx = float(np.max(binned_ratemap))
        ratemaps_norm[i, :] = (binned_ratemap.ravel() / mx) if mx > 0 else 0

    return si_values, avg_rate_values, z_scores, p_values, ratemaps, ratemaps_norm


def run_placecell_pipeline_one(mouse: str, session: str, frame_interval: float, overwrite: bool = False):
    """Run the complete place-cell pipeline for one mouse and one session."""
    assert session in ("pre", "post")

    save_path = output_dir(mouse, session)
    save_path.mkdir(parents=True, exist_ok=True)

    final_output_file = save_path / "p_values.csv"
    if final_output_file.exists() and not overwrite:
        print(f"[SKIP] {mouse} {session}: already processed -> {save_path}")
        return

    print(f"[RUN] {mouse} {session}")

    F1, coords, velocity, reward_trace, trace_time, cell_ids = load_and_align_session(
        mouse=mouse,
        session=session,
        frame_interval=frame_interval,
    )

    # Save aligned behaviour arrays.
    np.savetxt(save_path / "coords_trim.csv", coords, delimiter=",")
    np.savetxt(save_path / "velocity_trim.csv", velocity, delimiter=",")
    np.savetxt(save_path / "reward_trace_trim.csv", reward_trace.astype(int), delimiter=",")
    np.savetxt(save_path / "trace_time_trim.csv", trace_time, delimiter=",")

    reward_onsets = np.where(np.diff(reward_trace.astype(int), prepend=0) == 1)[0]
    reward_pos = coords[reward_onsets]
    np.savetxt(save_path / "reward_pos.csv", reward_pos, delimiter=",")

    # Baseline correction.
    F1_offset = F1 + FLUORESCENCE_OFFSET
    Fc = Parallel(n_jobs=-1)(
        delayed(baseline_correction)(trace, BASELINE_WINDOW_FRAMES, BASELINE_PERCENTILE)
        for trace in tqdm(F1_offset, desc=f"{mouse} {session} baseline")
    )
    Fc = np.asarray(Fc)

    Fc_df = pd.DataFrame(Fc.T)
    Fc_df.columns = cell_ids
    Fc_df.to_csv(save_path / "Fc.csv", index=False)

    # Reload from disk to keep the workflow aligned with the original notebook.
    Fc_df = pd.read_csv(save_path / "Fc.csv")
    Fc = np.asarray(Fc_df.T)
    Fc_ids = np.asarray(Fc_df.columns)

    # Significant transients.
    Fc3 = get_significant_transients(Fc)

    Fc3_df = pd.DataFrame(Fc3.T)
    Fc3_df.columns = Fc_ids
    Fc3_df.to_csv(save_path / "Fc3.csv", index=False)

    # SNR-based anomaly removal.
    snr = calculate_mad_snr(Fc3, Fc)
    snr_df = pd.DataFrame(snr)
    anomalies = list(snr_df[snr_df[0] > SNR_THRESHOLD].index)

    Fc3_cleaned = np.delete(Fc3, anomalies, axis=0)
    Fc_cleaned = np.delete(Fc, anomalies, axis=0)
    cell_ids_cleaned = np.delete(Fc_ids, anomalies, axis=0)

    Fc3_clean_df = pd.DataFrame(Fc3_cleaned.T)
    Fc3_clean_df.columns = cell_ids_cleaned
    Fc3_clean_df.to_csv(save_path / "Fc3_cleaned.csv", index=False)

    Fc_clean_df = pd.DataFrame(Fc_cleaned.T)
    Fc_clean_df.columns = cell_ids_cleaned
    Fc_clean_df.to_csv(save_path / "Fc_cleaned.csv", index=False)

    # Place-cell analysis.
    si_values, avg_rate_values, z_scores, p_values, ratemaps, ratemaps_norm = compute_place_cell_metrics(
        Fc3_cleaned=Fc3_cleaned,
        coords=coords,
        velocity=velocity,
        trace_time=trace_time,
    )

    calculate_placecell_pcnt(p_values, alpha=PLACE_CELL_ALPHA)

    # Save outputs.
    np.savetxt(save_path / "si_values.csv", si_values, delimiter=",")
    np.savetxt(save_path / "avg_rate_values.csv", avg_rate_values, delimiter=",")
    np.savetxt(save_path / "z_scores.csv", z_scores, delimiter=",")
    np.savetxt(save_path / "p_values.csv", p_values, delimiter=",")
    np.savetxt(save_path / "ratemaps.csv", ratemaps, delimiter=",")
    np.savetxt(save_path / "ratemaps_norm.csv", ratemaps_norm, delimiter=",")
    np.savetxt(save_path / "cell_ids.csv", cell_ids_cleaned, delimiter=",", fmt="%s")

    # Plot significant place cells only.
    sig_mask = (p_values <= PLACE_CELL_ALPHA) & (ratemaps_norm.max(axis=1) > 0)
    place_cell_rates = ratemaps_norm[sig_mask, :]
    place_cell_ids = cell_ids_cleaned[sig_mask]

    print(f"[PC] {place_cell_rates.shape[0]} significant place cells ({mouse} {session})")

    if place_cell_rates.shape[0] > 0:
        peak_bins = np.argmax(place_cell_rates, axis=1)
        sort_idx = np.argsort(peak_bins)

        sorted_place_cell_rates = place_cell_rates[sort_idx, :]
        sorted_place_cell_ids = place_cell_ids[sort_idx]

        np.savetxt(
            save_path / "place_cell_ids_sig_sorted.csv",
            sorted_place_cell_ids,
            delimiter=",",
            fmt="%s",
        )

        save_ratemap_figure(
            ratemaps_norm=sorted_place_cell_rates,
            save_path=save_path,
            session=session,
            mouse=mouse,
            reward_pos=reward_pos,
            track_length_cm=TRACK_LENGTH_CM,
            filename=f"{mouse}_{session}_place_cell_ratemaps_sig.png",
            show=False,
        )

    print(f"[DONE] {mouse} {session} -> {save_path}")

## 7. Batch runner

This section computes a frame interval for each mouse using the full-session behaviour file, then processes the pre- and post-switch sessions for each mouse.

In [ ]:
def run_batch(mice, overwrite=False):
    """Run pre/post place-cell processing for all mice."""
    frame_intervals = {}

    print("---- computing frame intervals ----")
    for mouse in mice:
        try:
            frame_intervals[mouse] = compute_frame_interval(mouse)
        except Exception as e:
            print(f"[ERROR] {mouse}: could not compute frame interval: {e}")

    print("---- running pre/post sessions ----")
    for mouse in mice:
        if mouse not in frame_intervals:
            continue

        for session in SESSIONS:
            try:
                run_placecell_pipeline_one(
                    mouse=mouse,
                    session=session,
                    frame_interval=frame_intervals[mouse],
                    overwrite=overwrite,
                )
            except Exception as e:
                print(f"[ERROR] {mouse} {session}: {e}")

## 8. Run the analysis

Set `OVERWRITE = True` if you want to regenerate outputs that already exist.

In [ ]:
run_batch(MICE, overwrite=OVERWRITE)
